# Case 1 — Cylinder Flow: Lift Coefficient ($C_l$) POD-AS-PRS Workflow

This notebook reproduces the lift-coefficient ($C_l$) surrogate study for a
2-D cylinder at $Re = 100$ using the full POD-AS-PRS pipeline:

1. **POD** – decompose vorticity snapshots via SVD
2. **ResNet** – train a fully-connected ResNet to map POD coefficients → $C_l$
3. **Gradient analysis** – validate autograd against finite difference
4. **Active Subspaces (AS)** – identify the dominant input directions
5. **Polynomial Response Surface (PRS)** – fit and evaluate the low-dimensional surrogate
6. **Sensitivity comparison** – compare AS activity scores vs Pearson $|r|$ vs standardised $|beta|$

## 0 · Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

from core.pod_engine       import POD_SVD
from core.resnet_model     import ResNet
from core.resnet_trainer   import (set_random_seed, load_or_train,
                                    evaluate_and_save_metrics,
                                    plot_loss_curve, plot_prediction_comparison,
                                    compute_all_gradients)
from core.gradient_analysis import compare_gradients_nature_style_dataset
from utils.data_loader      import (load_and_preprocess_data, denormalise,
                                     load_pod_vis_data)
from utils.visualization    import (plot_pod_importance, plot_response_surface_2d,
                                     validate_response_surface,
                                     compare_rom_fom_predictions,
                                     plot_polynomial_cv,
                                     plot_subspace_polynomial_heatmap,
                                     plot_interaction_heatmap,
                                     plot_pod_energy,
                                     plot_eigenvalues,
                                     plot_pod_modes_and_coeffs,
                                     plot_pod_phase_space_triangle,
                                     plot_mesh_and_vorticity,
                                     plot_qoi)
import lib.active_subspaces as ac

set_random_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Random seed set to: 42
Device: cuda


## 1 · Data Paths
Adjust these paths to match your local directory layout.

In [2]:
FLOW_DATA_PATH = '../data/Case1_Cylinder/flow_field_data.npz'
QOI_DATA_PATH  = '../data/Case1_Cylinder/lift_coefficient.dat'
RESULTS_DIR    = '../results/Case1_Cylinder_Cl'
MODEL_PATH     = os.path.join(RESULTS_DIR, 'resnet_model.pth')

NUM_POD_COEFFS = 150
os.makedirs(RESULTS_DIR, exist_ok=True)

## 2 · Load Data and Run POD

In [ ]:
# Reuse the POD decomposition already computed by Case1_Cylinder (same flow field)
POD_CACHE_DIR = '../results/Case1_Cylinder/POD'

(
    train_loader, val_loader, test_loader,
    pod_coeffs, pod_coeffs_norm,
    pod_mean, pod_std,
    qoi_mean, qoi_std,
    pod_min, pod_max,
    qoi_min, qoi_max,
) = load_and_preprocess_data(
    flow_data_path=FLOW_DATA_PATH,
    qoi_data_path=QOI_DATA_PATH,
    num_pod_coeffs=NUM_POD_COEFFS,
    train_ratio=0.8,
    val_ratio=0.1,
    batch_size=32,
    apply_region_filter=False,
    pod_save_dir=POD_CACHE_DIR,
)

print(f'POD coefficients shape : {pod_coeffs.shape}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')
print(f'POD/mesh visualisations: see ../results/Case1_Cylinder/')

pod_vis = load_pod_vis_data(
    pod_save_dir=POD_CACHE_DIR,
    flow_data_path=FLOW_DATA_PATH,
    apply_region_filter=False,
)

plot_eigenvalues(
    pod_vis['S'], num_values=20,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='cylinder',
)


Loading flow field data...
Flow data keys: ['coords', 'times', 'velocity', 'pressure', 'vorticity', 'vorticity_grid_x', 'vorticity_grid_y', 'n_timesteps', 'n_elements', 'n_components', 'n_points', 'start_time', 'end_time', 'start_idx', 'end_idx', 'mesh_limits_x', 'mesh_limits_y', 'vorticity_nx', 'vorticity_ny']


Vorticity array shape: (1000, 356, 593)
Grid range: X=[-15.00, 35.00], Y=[-15.00, 15.00]
Reshaped vorticity: (1000, 211108)
Loading cached POD data from: ../results/Case1_Cylinder/POD/pod_data.npz


## 3 · Build and Train the ResNet Surrogate

In [ ]:
set_random_seed(42)   


NUM_BLOCKS = 7
model = ResNet(input_size=NUM_POD_COEFFS, hidden_size=128,
               num_blocks=NUM_BLOCKS, dropout_rate=0.1).to(DEVICE)
print(model)

model, train_losses, val_losses = load_or_train(
    model, train_loader, val_loader, DEVICE,
    model_save_path=MODEL_PATH,
    num_epochs=1000,
    patience=100,
    lr=0.001,
)

plot_loss_curve(train_losses, val_losses, patience=100,
                results_dir=RESULTS_DIR)

Random seed set to: 42
ResNet(
  (input_layer): Sequential(
    (0): Linear(in_features=150, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res_blocks): ModuleList(
    (0-6): 7 x ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.1, inplace=False)
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU()
    )
  )
  (output_layer): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.1, in


Starting training (max 1000 epochs, patience=100)...
Epoch 10/1000 | Train Loss: 0.1182 | Val Loss: 0.1500 | Patience: 0/100
Epoch 20/1000 | Train Loss: 0.0704 | Val Loss: 0.0864 | Patience: 0/100
Epoch 30/1000 | Train Loss: 0.0453 | Val Loss: 0.0491 | Patience: 2/100
Epoch 40/1000 | Train Loss: 0.0400 | Val Loss: 0.0462 | Patience: 4/100
Epoch 50/1000 | Train Loss: 0.0409 | Val Loss: 0.0216 | Patience: 1/100
Epoch 60/1000 | Train Loss: 0.0344 | Val Loss: 0.0193 | Patience: 1/100
Epoch 70/1000 | Train Loss: 0.0281 | Val Loss: 0.0119 | Patience: 0/100
Epoch 80/1000 | Train Loss: 0.0217 | Val Loss: 0.0090 | Patience: 8/100
Epoch 90/1000 | Train Loss: 0.0203 | Val Loss: 0.0106 | Patience: 7/100
Epoch 100/1000 | Train Loss: 0.0328 | Val Loss: 0.0065 | Patience: 3/100
Epoch 110/1000 | Train Loss: 0.0218 | Val Loss: 0.0044 | Patience: 0/100
Epoch 120/1000 | Train Loss: 0.0213 | Val Loss: 0.0047 | Patience: 8/100
Epoch 130/1000 | Train Loss: 0.0149 | Val Loss: 0.0049 | Patience: 5/100
Epoch 

## 4 · Evaluate the Surrogate

In [ ]:
denorm = lambda v: denormalise(v, qoi_mean, qoi_std)

metrics = evaluate_and_save_metrics(
    model,
    loaders=[train_loader, val_loader, test_loader],
    split_names=['Train', 'Validation', 'Test'],
    device=DEVICE,
    denorm_fn=denorm,
    results_dir=RESULTS_DIR,
)

Train: MSE=3.808303e-05, MAE=5.001807e-03, R²=0.999310, MaxRelErr=22.174549
Validation: MSE=1.936096e-04, MAE=1.140736e-02, R²=0.996593, MaxRelErr=2.740712
Test: MSE=1.547822e-04, MAE=9.515494e-03, R²=0.996377, MaxRelErr=8.687518
Metrics saved to ../results/Case1_Cylinder_Cl/metrics.txt


## 5 · Gradient Analysis (Autograd vs Finite Difference)

In [ ]:
# Collect only training-set samples — matches legacy train.py lines 344-354
# (iterates train_loader only, not all 1000 samples)
pod_train_list = []
for inputs, _ in train_loader:
    pod_train_list.append(inputs)
pod_norm_torch = torch.cat(pod_train_list, dim=0)   # shape: (800, NUM_POD_COEFFS)

avg_err, max_err, timing = compare_gradients_nature_style_dataset(
    model, pod_norm_torch, device=DEVICE,
    h=1e-2, max_modes=20,
    save_path=os.path.join(RESULTS_DIR, 'gradient_comparison.pdf'),
    batch_size=16,   # matches legacy train.py line 368
)
print(f'Mean rel. error: {avg_err:.6f}, Max rel. error: {max_err:.6f}')
print(f'AD/FD speedup: {timing["speedup_ratio"]:.1f}x')


Dataset-level gradient comparison for 1000 samples...
  Batch 1/32 (samples 0–31)


  Batch 2/32 (samples 32–63)
  Batch 3/32 (samples 64–95)
  Batch 4/32 (samples 96–127)
  Batch 5/32 (samples 128–159)
  Batch 6/32 (samples 160–191)
  Batch 7/32 (samples 192–223)
  Batch 8/32 (samples 224–255)
  Batch 9/32 (samples 256–287)
  Batch 10/32 (samples 288–319)
  Batch 11/32 (samples 320–351)
  Batch 12/32 (samples 352–383)
  Batch 13/32 (samples 384–415)
  Batch 14/32 (samples 416–447)
  Batch 15/32 (samples 448–479)
  Batch 16/32 (samples 480–511)
  Batch 17/32 (samples 512–543)
  Batch 18/32 (samples 544–575)
  Batch 19/32 (samples 576–607)
  Batch 20/32 (samples 608–639)
  Batch 21/32 (samples 640–671)
  Batch 22/32 (samples 672–703)
  Batch 23/32 (samples 704–735)
  Batch 24/32 (samples 736–767)
  Batch 25/32 (samples 768–799)
  Batch 26/32 (samples 800–831)
  Batch 27/32 (samples 832–863)
  Batch 28/32 (samples 864–895)
  Batch 29/32 (samples 896–927)
  Batch 30/32 (samples 928–959)
  Batch 31/32 (samples 960–991)
  Batch 32/32 (samples 992–999)

=== Gradient Timing 

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/core/gradient_analysis.py:327: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x="POD Mode", y="Gradient Difference (AD - FD)",


Mean rel. error: 0.064824, Max rel. error: 100.000000
AD/FD speedup: 90.4x


## 6 · Compute Gradients for All Samples

In [ ]:
gradients = compute_all_gradients(model, pod_coeffs, DEVICE, batch_size=32)
print(f'Gradient matrix shape: {gradients.shape}')

grad_save = os.path.join(RESULTS_DIR, 'pod_gradients.npy')
np.save(grad_save, gradients)
print(f'Saved to {grad_save}')

Normalised POD range: [-1.0000, 1.0000]
  Gradient batch 1/32 (samples 0–31)
  Gradient batch 2/32 (samples 32–63)
  Gradient batch 3/32 (samples 64–95)
  Gradient batch 4/32 (samples 96–127)
  Gradient batch 5/32 (samples 128–159)
  Gradient batch 6/32 (samples 160–191)
  Gradient batch 7/32 (samples 192–223)
  Gradient batch 8/32 (samples 224–255)
  Gradient batch 9/32 (samples 256–287)
  Gradient batch 10/32 (samples 288–319)
  Gradient batch 11/32 (samples 320–351)
  Gradient batch 12/32 (samples 352–383)
  Gradient batch 13/32 (samples 384–415)
  Gradient batch 14/32 (samples 416–447)
  Gradient batch 15/32 (samples 448–479)
  Gradient batch 16/32 (samples 480–511)
  Gradient batch 17/32 (samples 512–543)
  Gradient batch 18/32 (samples 544–575)
  Gradient batch 19/32 (samples 576–607)
  Gradient batch 20/32 (samples 608–639)
  Gradient batch 21/32 (samples 640–671)
  Gradient batch 22/32 (samples 672–703)
  Gradient batch 23/32 (samples 704–735)
  Gradient batch 24/32 (samples 73

## 7 · Active Subspace Analysis

In [ ]:
# All-sample min/max for gradient scaling — matches legacy main.py lines 27-46
XX_as_min = np.min(pod_coeffs, axis=0)
XX_as_max = np.max(pod_coeffs, axis=0)
scale = (XX_as_max - XX_as_min) / 2.0
scale[scale < 1e-10] = 1.0
gradients_scaled = gradients * scale

# Bootstrap-based AS computation — nboot=1000 matches legacy Example1_Cylinder_Cl/main.py
ss = ac.subspaces.Subspaces()
ss.compute(df=gradients_scaled, nboot=1000)

# Plot first 6 eigenvalues, subspace errors, and eigenvectors
opts = ac.utils.plotters.plot_opts(savefigs=True)
ac.utils.plotters.eigenvalues(
    ss.eigenvals[:6],
    e_br=ss.e_br[:6, :],
    out_label='$C_l$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvalues.jpg'),
)
ac.utils.plotters.subspace_errors(
    ss.sub_br[:6, :], out_label='$C_l$', opts=opts
)
ac.utils.plotters.eigenvectors(
    ss.eigenvecs[:6, :2],
    out_label='$C_l$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvectors.jpg'),
)

# Partition: use the 2-D active subspace
n_active = 2
ss.partition(n_active)
print(f'Active subspace dimension: {n_active}')
print(f'W1 shape: {ss.W1.shape}')

Active subspace dimension: 2
W1 shape: (150, 2)


## 8 · Sufficient Summary Plot and Mode Importance

In [ ]:
# Load lift coefficient (raw physical values)
qoi_raw = np.loadtxt(QOI_DATA_PATH)
qoi_full = qoi_raw[:pod_coeffs.shape[0], 1]

# Project onto active subspace (XX_as_min/max from Cell 20 — all-sample min/max)
pod_norm_all = 2.0 * (pod_coeffs - XX_as_min) / (XX_as_max - XX_as_min) - 1.0
y_active = pod_norm_all @ ss.W1

ac.utils.plotters.sufficient_summary(
    y_active, qoi_full.reshape(-1, 1),
    out_label='$C_l$', opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'sufficient_summary.jpg'),
)

# POD importance — weighted sum then normalise (matches legacy main.py lines 94-110)
pod_importance = np.sum(ss.eigenvecs[:, :n_active] ** 2 * ss.eigenvals[:n_active, 0], axis=1)
total = pod_importance.sum()
if total > 0:
    pod_importance = pod_importance / total
else:
    pod_importance = np.ones(NUM_POD_COEFFS) / NUM_POD_COEFFS

plot_pod_importance(
    NUM_POD_COEFFS, pod_importance,
    save_dir=os.path.join(RESULTS_DIR, 'Importance'),
    top_n=6,
)

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/utils/visualization.py:178: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax.grid(False, axis='x', alpha=0.3)


'../results/Case1_Cylinder_Cl/Importance/pod_mode_importance_150_top6.jpg'

## 9 · Subspace–Polynomial R² Heatmap

In [ ]:
# Select the 2 most important POD modes (matches legacy main.py lines 131-157)
n_dim_range  = 2
n_poly_range = 3
n_importance = np.argsort(pod_importance)[-n_dim_range:][::-1]

XX_as_23      = pod_norm_all[:, n_importance]      # (N, 2)
eigenvecs_23  = ss.eigenvecs[n_importance, :]      # (2, k)

heatmap_path, heatmap_data = plot_subspace_polynomial_heatmap(
    XX_as_23,
    qoi_full.reshape(-1, 1),
    eigenvecs_23,
    n_dim_range=n_dim_range,
    n_poly_range=n_poly_range,
    save_dir=os.path.join(RESULTS_DIR, 'Heatmap'),
)

r2_matrix = heatmap_data['r2_matrix']
best_idx          = np.unravel_index(np.nanargmax(r2_matrix), r2_matrix.shape)
best_dim, best_poly = best_idx[0] + 1, best_idx[1] + 1
print(f'Heatmap saved to {heatmap_path}')
print(f'Best R²={r2_matrix[best_idx]:.6f}  →  dim={best_dim}, poly_order={best_poly}')
print('R² matrix:\n', r2_matrix)

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

Heatmap saved to ../results/Case1_Cylinder_Cl/Heatmap/subspace_polynomial_heatmap.jpg
Best R²=0.999981  →  dim=2, poly_order=3
R² matrix:
 [[0.99668653 0.99886514 0.99885607]
 [0.999865   0.99994344 0.99998111]]


## 10 · Activity Scores and Interaction Heatmap

In [ ]:
# Activity scores for the first 6 modes (matches legacy main.py lines 272-292)
top_n    = 6
alpha_D  = (ss.eigenvals[:n_active].reshape(1, n_active) * ss.eigenvecs[:, :n_active] ** 2).sum(axis=1)
alpha_D_top = alpha_D[:top_n]
print(f'Activity scores (first {top_n}):', alpha_D_top)
print('Normalised:                      ', alpha_D_top / alpha_D_top.sum())

# Lower-triangle modal interaction heatmap (matches legacy main.py lines 296-389)
heatmap_fig = plot_interaction_heatmap(
    ss.eigenvecs,
    ss.eigenvals,
    pod_importance,
    n_active=n_active,
    top_n=top_n,
    save_dir=os.path.join(RESULTS_DIR, 'Activity_Score'),
)
print(f'Interaction heatmap saved to {heatmap_fig}')

Activity scores (first 6): [3.22705604e-02 3.02924617e+01 3.08558213e-01 7.61317595e-01
 2.70362678e-02 1.18380208e-01]
Normalised:                       [1.02316218e-03 9.60445090e-01 9.78306826e-03 2.41381421e-02
 8.57205032e-04 3.75333278e-03]


Interaction heatmap saved to ../results/Case1_Cylinder_Cl/Activity_Score/param_interaction_heatmap_lower_top6.jpg


## 11 · Polynomial Response Surface

In [ ]:
from lib.active_subspaces.utils.rs import PolynomialApproximation
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Project onto best_dim-dimensional reduced subspace (matches legacy main.py line 178)
y_23 = XX_as_23.dot(ss.eigenvecs[n_importance, :best_dim])

X_train, X_test, f_train, f_test = train_test_split(
    y_23, qoi_full.reshape(-1, 1), test_size=0.2, random_state=42
)

# Cross-validate polynomial order 1–8 (matches legacy main.py lines 194-211)
n_values, r2_values, rmse_values = [], [], []
best_cv_score, best_cv_rmse, best_cv_n = -1, float('inf'), 1

for n in range(1, 4):
    rs_cv = PolynomialApproximation(N=n)
    rs_cv.train(X_train, f_train)
    pred  = rs_cv.predict(X_test)[0]
    score = r2_score(f_test, pred)
    rmse  = np.sqrt(mean_squared_error(f_test, pred))
    print(f'N={n}: R²={score:.6f}, RMSE={rmse:.8f}')
    n_values.append(n); r2_values.append(score); rmse_values.append(rmse)
    if score > best_cv_score or (abs(score - best_cv_score) < 1e-4 and rmse < best_cv_rmse):
        best_cv_score, best_cv_rmse, best_cv_n = score, rmse, n

print(f'\nBest poly order (CV): N={best_cv_n}, R²={best_cv_score:.6f}')

plot_polynomial_cv(
    n_values, r2_values, rmse_values, best_cv_n, best_cv_score, best_cv_rmse,
    save_dir=os.path.join(RESULTS_DIR, 'Polynomial_CV'),
)

# Train final RS using best_poly from R² heatmap
RS = PolynomialApproximation(N=best_poly)
RS.train(X_train, f_train)
print(f'Final RS trained with N={best_poly}, train R²={RS.Rsqr:.6f}')

# Build 2-D grid for visualisation
xx, yy = np.meshgrid(
    np.linspace(y_23[:, 0].min(), y_23[:, 0].max(), 50),
    np.linspace(y_23[:, 1].min(), y_23[:, 1].max(), 50),
)
n_dims      = y_23.shape[1]
grid_full   = np.zeros((xx.size, n_dims))
grid_full[:, :2] = np.column_stack([xx.ravel(), yy.ravel()])
zz = RS.predict(grid_full)[0].reshape(xx.shape)

plot_response_surface_2d(
    xx, yy, zz, y_23, qoi_full,
    results_dir=os.path.join(RESULTS_DIR, 'PRS'),
)

validate_response_surface(
    RS, X_test, f_test,
    save_dir=os.path.join(RESULTS_DIR, 'RS_Validation'),
)

compare_rom_fom_predictions(
    y_23, qoi_full, RS,
    t_start=250.0, dt=0.05,
    qoi_label='$C_l$',
    geometry='cylinder',
    save_dir=os.path.join(RESULTS_DIR, 'ROM_FOM'),
)

N=1: R²=0.999865, RMSE=0.00272386
N=2: R²=0.999943, RMSE=0.00176304
N=3: R²=0.999981, RMSE=0.00101890
N=4: R²=0.999988, RMSE=0.00081901
N=5: R²=0.999991, RMSE=0.00068685
N=6: R²=0.999994, RMSE=0.00059523
N=7: R²=0.999995, RMSE=0.00052639
N=8: R²=0.999996, RMSE=0.00046862

Best poly order (CV): N=8, R²=0.999996


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/utils/visualization.py:251: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]


Final RS trained with N=3, train R²=0.999980

Test samples: 200
  R²   = 0.999981
  RMSE = 0.00101890
  MAE  = 0.00073062
Metrics saved to: ../results/Case1_Cylinder_Cl/RS_Validation/validation_metrics.txt
ROM vs FOM: R²=1.0000, RMSE=0.001037, MAE=0.268275


{'r2': 0.9999803388097636,
 'rmse': 0.001037137492390451,
 'mae': 0.26827482006849696}

## 12 · Sensitivity Comparison: AS vs Pearson vs Standardised $|beta|$

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr
from sklearn.linear_model import LinearRegression

f_flat = qoi_full.flatten()  # (N,)

# ── Method 1: Pearson |r| ────────────────────────────────────────────────────
correlations = np.array([stats.pearsonr(pod_norm_all[:, i], f_flat)[0]
                         for i in range(NUM_POD_COEFFS)])
abs_corr = np.abs(correlations)

# ── Method 2: Standardised regression coefficient |β| ────────────────────────
lr      = LinearRegression().fit(pod_norm_all, f_flat)
sigma_f = np.std(f_flat)
abs_beta = np.abs(lr.coef_ * np.sqrt(1.0 / 3.0) / sigma_f)

# ── Ranking comparison ────────────────────────────────────────────────────────
as_ranking   = np.argsort(pod_importance)[::-1]
corr_ranking = np.argsort(abs_corr)[::-1]
beta_ranking = np.argsort(abs_beta)[::-1]

print('AS activity score   top-10 modes:', as_ranking[:10] + 1)
print('Pearson |r|         top-10 modes:', corr_ranking[:10] + 1)
print('Std. regression |β| top-10 modes:', beta_ranking[:10] + 1)

# ── Spearman rank correlations ────────────────────────────────────────────────
rho_as_corr,   _ = spearmanr(pod_importance, abs_corr)
rho_as_beta,   _ = spearmanr(pod_importance, abs_beta)
rho_corr_beta, _ = spearmanr(abs_corr, abs_beta)
print(f'\nSpearman ρ  AS vs |r|:  {rho_as_corr:.4f}')
print(f'Spearman ρ  AS vs |β|:  {rho_as_beta:.4f}')
print(f'Spearman ρ  |r| vs |β|: {rho_corr_beta:.4f}')

# ── Numeric table: first 6 modes ─────────────────────────────────────────────
top_n_compare = 6
print(f'\n{"Mode":>8} {"AS score":>12} {"AS rank":>9} '
      f'{"Pearson |r|":>13} {"r rank":>8} {"beta":>10} {"β rank":>8}')
print('-' * 76)
for i in range(top_n_compare):
    ar  = np.where(as_ranking   == i)[0][0] + 1
    cr  = np.where(corr_ranking == i)[0][0] + 1
    br  = np.where(beta_ranking == i)[0][0] + 1
    print(f'{"Mode "+str(i+1):>8} {pod_importance[i]:>12.6f} {ar:>9d} '
          f'{abs_corr[i]:>13.6f} {cr:>8d} {abs_beta[i]:>10.6f} {br:>8d}')

# ── Line-plot comparison: first 6 modes ──────────────────────────────────────
corr_dir  = os.path.join(RESULTS_DIR, 'Correlation_Comparison')
os.makedirs(corr_dir, exist_ok=True)

x_vals      = np.arange(top_n_compare)
mode_labels = [f'Mode {i+1}' for i in range(top_n_compare)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_vals, pod_importance[:top_n_compare], color='steelblue',
        marker='o', markersize=9, linewidth=2.0,
        label=r'AS activity score $\alpha_i$')
ax.plot(x_vals, abs_corr[:top_n_compare], color='coral',
        marker='s', markersize=9, linewidth=2.0,
        label=r'Pearson $|r_i|$')
ax.plot(x_vals, abs_beta[:top_n_compare], color='mediumseagreen',
        marker='^', markersize=9, linewidth=2.0,
        label=r'Standardised regression $|\beta_i|$')
ax.set_ylabel('Sensitivity metric value', fontsize=24)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(x_vals)
ax.set_xticklabels(mode_labels, fontsize=22)
ax.set_xlabel('POD mode', fontsize=24)
ax.tick_params(axis='y', labelsize=20)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.grid(axis='x', linestyle='--', alpha=0.2)
ax.legend(fontsize=18, frameon=False, loc='upper right')
plt.tight_layout()
fig_path = os.path.join(corr_dir, 'AS_vs_Pearson_vs_Beta_top6_lineplot.pdf')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.savefig(fig_path.replace('.pdf', '.jpg'), dpi=300, bbox_inches='tight')
plt.close()
print(f'\n3-way comparison plot saved to {fig_path}')

AS activity score   top-10 modes: [ 2  4  3  6  1  5  8 10  9  7]
Pearson |r|         top-10 modes: [ 2  1  3  6  4  5 10  7 30  9]
Std. regression |β| top-10 modes: [ 2  1  3  6  4  5 30 10  7 29]

Spearman ρ  AS vs |r|:  0.3476
Spearman ρ  AS vs |β|:  0.1854
Spearman ρ  |r| vs |β|: 0.9355

    Mode     AS score   AS rank   Pearson |r|   r rank       beta   β rank
----------------------------------------------------------------------------
  Mode 1     0.001023         5      0.010496        2   0.008534        2
  Mode 2     0.960367         1      0.999931        1   0.819924        1
  Mode 3     0.009782         3      0.003904        3   0.003237        3
  Mode 4     0.024136         2      0.001152        5   0.000964        5
  Mode 5     0.000857         6      0.000974        6   0.000853        6
  Mode 6     0.003753         4      0.003182        4   0.002819        4

3-way comparison plot saved to ../results/Case1_Cylinder_Cl/Correlation_Comparison/AS_vs_Pearson_vs_Beta

## 13 · ROM Selection Comparison: AS-selected vs Pearson-selected Modes

In [ ]:
rom_dir = os.path.join(RESULTS_DIR, 'ROM_Selection_Comparison')
os.makedirs(rom_dir, exist_ok=True)

# ── Select top-2 modes by each method ────────────────────────────────────────
as_modes   = np.argsort(pod_importance)[::-1][:2]
corr_modes = np.argsort(abs_corr)[::-1][:2]
print(f'AS-selected modes:      {as_modes + 1}')
print(f'Pearson-selected modes: {corr_modes + 1}')

y_as   = pod_norm_all[:, as_modes].dot(ss.eigenvecs[as_modes,   :best_dim])
y_corr = pod_norm_all[:, corr_modes].dot(ss.eigenvecs[corr_modes, :best_dim])

# ── Train/test split (same seed) ─────────────────────────────────────────────
X_tr_as,   X_te_as,   f_tr_as,   f_te_as   = train_test_split(
    y_as,   qoi_full.reshape(-1, 1), test_size=0.2, random_state=42)
X_tr_corr, X_te_corr, f_tr_corr, f_te_corr = train_test_split(
    y_corr, qoi_full.reshape(-1, 1), test_size=0.2, random_state=42)

# ── Helper: CV + train best RS ───────────────────────────────────────────────
def train_best_rs(X_tr, f_tr, X_te, f_te, max_n=8):
    best_s, best_n, best_r = -1, 1, float('inf')
    for n in range(1, max_n + 1):
        rs = PolynomialApproximation(N=n)
        rs.train(X_tr, f_tr)
        pred  = rs.predict(X_te)[0]
        s     = r2_score(f_te, pred)
        r     = np.sqrt(mean_squared_error(f_te, pred))
        if s > best_s or (abs(s - best_s) < 1e-4 and r < best_r):
            best_s, best_n, best_r = s, n, r
    rs_f = PolynomialApproximation(N=best_n)
    rs_f.train(X_tr, f_tr)
    return rs_f, best_n, best_s, best_r

RS_as,   n_as,   r2_as_te,   rmse_as_te   = train_best_rs(X_tr_as,   f_tr_as,   X_te_as,   f_te_as)
RS_corr, n_corr, r2_corr_te, rmse_corr_te = train_best_rs(X_tr_corr, f_tr_corr, X_te_corr, f_te_corr)

pred_te_as   = RS_as.predict(X_te_as)[0]
pred_te_corr = RS_corr.predict(X_te_corr)[0]
mae_as_te    = np.mean(np.abs(f_te_as   - pred_te_as))
mae_corr_te  = np.mean(np.abs(f_te_corr - pred_te_corr))

print(f'\nROM-AS   (N={n_as}):   Test R²={r2_as_te:.6f}, RMSE={rmse_as_te:.8f}, MAE={mae_as_te:.8f}')
print(f'ROM-Corr (N={n_corr}): Test R²={r2_corr_te:.6f}, RMSE={rmse_corr_te:.8f}, MAE={mae_corr_te:.8f}')
print(f'ΔR²  (AS−Corr): {r2_as_te - r2_corr_te:+.6f}')

# ── Scatter: test-set true vs predicted ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, f_t, f_p, title, color in [
    (axes[0], f_te_as.flatten(),   pred_te_as.flatten(),
     f'ROM-AS (Mode {list(as_modes+1)})\nTest $R^2$={r2_as_te:.5f}', 'steelblue'),
    (axes[1], f_te_corr.flatten(), pred_te_corr.flatten(),
     f'ROM-Corr (Mode {list(corr_modes+1)})\nTest $R^2$={r2_corr_te:.5f}', 'coral'),
]:
    ax.scatter(f_t, f_p, color=color, alpha=0.5, s=15)
    lims = [min(f_t.min(), f_p.min()) - 0.01, max(f_t.max(), f_p.max()) + 0.01]
    ax.plot(lims, lims, 'k--', linewidth=1.5)
    ax.set_xlabel('FOM $C_l$', fontsize=20)
    ax.set_ylabel('ROM prediction', fontsize=20)
    ax.set_title(title, fontsize=14)
    ax.tick_params(labelsize=16)
    ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
sc_path = os.path.join(rom_dir, 'ROM_AS_vs_Corr_testset_scatter.pdf')
plt.savefig(sc_path, dpi=300, bbox_inches='tight')
plt.savefig(sc_path.replace('.pdf', '.jpg'), dpi=300, bbox_inches='tight')
plt.close()

# ── Error distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.abs(f_te_as.flatten()   - pred_te_as.flatten()),   bins=30,
        alpha=0.6, color='steelblue', density=True, label=f'ROM-AS   MAE={mae_as_te:.6f}')
ax.hist(np.abs(f_te_corr.flatten() - pred_te_corr.flatten()), bins=30,
        alpha=0.6, color='coral',     density=True, label=f'ROM-Corr MAE={mae_corr_te:.6f}')
ax.set_xlabel('Absolute prediction error (test set)', fontsize=18)
ax.set_ylabel('Density', fontsize=18)
ax.tick_params(labelsize=14)
ax.legend(fontsize=13, frameon=False)
ax.grid(linestyle='--', alpha=0.4)
plt.tight_layout()
err_path = os.path.join(rom_dir, 'ROM_AS_vs_Corr_error_dist.pdf')
plt.savefig(err_path, dpi=300, bbox_inches='tight')
plt.savefig(err_path.replace('.pdf', '.jpg'), dpi=300, bbox_inches='tight')
plt.close()
print(f'Scatter plot saved to {sc_path}')
print(f'Error distribution saved to {err_path}')

AS-selected modes:      [2 4]
Pearson-selected modes: [2 1]

ROM-AS   (N=8):   Test R²=0.999996, RMSE=0.00046862, MAE=0.00024727
ROM-Corr (N=7): Test R²=1.000000, RMSE=0.00000279, MAE=0.00000214
ΔR²  (AS−Corr): -0.000004


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

Scatter plot saved to ../results/Case1_Cylinder_Cl/ROM_Selection_Comparison/ROM_AS_vs_Corr_testset_scatter.pdf
Error distribution saved to ../results/Case1_Cylinder_Cl/ROM_Selection_Comparison/ROM_AS_vs_Corr_error_dist.pdf
